# USGS: extracto actualizado de eventos a 150 km

Análisis independiente del archivo `usgs_replicas.geojson`. El nombre del archivo no demuestra que todos los eventos sean réplicas. Se conserva el principal y se identifica por ID antes de contar posteriores. La comparación con el extracto anterior corresponde a la síntesis posterior, ya que cambian radio y ventana temporal.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad de análisis

Archivo: `data/externos/usgs_replicas.geojson`. Una fila representa un evento instrumental localizado por USGS en la ventana y radio de consulta.
La fecha de descarga no equivale a fecha de verificación del objeto en terreno.

In [ ]:
PATH = ROOT / 'data/externos/usgs_replicas.geojson'
meta = ficha(PATH)
raw, df, g = leer_geojson(PATH)
print("Archivo:", meta["archivo"], "| filas:", len(df), "| propiedades:", len(df.columns))
display(pd.Series(meta, name="procedencia").to_frame())

## 2. Calidad por columna y geometría

In [ ]:
calidad = mostrar_perfil(df)
geo = geometria(g)
display(pd.Series(geo, name="geometria").to_frame())
resumen = {"dataset": '17_usgs_eventos_actualizados', **meta, "filas": len(df), "columnas": list(df.columns),
           "calidad_columnas": calidad.to_dict("records"), "geometria": geo}

## 3. Identidad, tiempo UTC y profundidad del GeoJSON 3D

In [ ]:
ids=pd.Series([f.get("id") for f in raw["features"]],name="evento_id")
fecha=pd.to_datetime(df.time,unit="ms",utc=True,errors="coerce")
actualizacion=pd.to_datetime(df.updated,unit="ms",utc=True,errors="coerce")
profundidad=pd.Series([f["geometry"]["coordinates"][2] if len(f["geometry"]["coordinates"])>2 else np.nan for f in raw["features"]])
magnitud=pd.to_numeric(df.mag,errors="coerce")
principal=df.mag.idxmax();fecha_principal=fecha.iloc[principal]
metadata=raw.get("metadata",{})
display(pd.Series({"consulta":metadata.get("url"),"filas":len(df),
 "eventos_id_nulos":int(ids.isna().sum()),"eventos_id_duplicados":int(ids.duplicated().sum()),
 "fecha_min_utc":str(fecha.min()),"fecha_max_utc":str(fecha.max()),
 "anteriores_al_mayor":int((fecha<fecha_principal).sum()),
 "posteriores_al_mayor":int((fecha>fecha_principal).sum()),
 "actualizacion_antes_de_origen":int((actualizacion<fecha).sum())}).to_frame("valor"))
display(pd.DataFrame({"magnitud":magnitud,"profundidad_km":profundidad}).describe())
for c in ["magType","type","status","net"]:display(conteos(df[c],c))
resumen.update({"metadata":metadata,"evento_id_duplicados":int(ids.duplicated().sum()),
 "fecha_min_utc":str(fecha.min()),"fecha_max_utc":str(fecha.max()),
 "mag_min":float(magnitud.min()),"mag_max":float(magnitud.max()),
 "profundidad_min_km":float(profundidad.min()),"profundidad_max_km":float(profundidad.max()),
 "mag_types":df.magType.value_counts().to_dict(),"posteriores_al_mayor":int((fecha>fecha_principal).sum())})

## 4. Distribución temporal y disponibilidad de intensidad

In [ ]:
mensual=fecha.dt.strftime("%Y-%m").value_counts().sort_index().rename_axis("mes_utc").reset_index(name="eventos")
display(mensual)
display(faltantes(df,["mag","magType","felt","cdi","mmi","alert","nst","rms","gap"]))
grafico_barras(mensual,"Eventos del extracto por mes UTC", "17_meses")
fig,axs=plt.subplots(1,2,figsize=(11,4));axs[0].hist(magnitud,bins=15,color="#226e86");axs[0].set(title="Magnitudes del extracto",xlabel="Magnitud reportada",ylabel="Eventos")
axs[1].hist(profundidad,bins=15,color="#b87932");axs[1].set(title="Profundidades",xlabel="km",ylabel="Eventos")
fig.tight_layout();fig.savefig(SALIDAS/"17_distribuciones.png",dpi=150);plt.show()
resumen["mensual"]=mensual.to_dict("records")
resumen["mmi_con_dato"]=int(df.mmi.notna().sum())
resumen["cdi_con_dato"]=int(df.cdi.notna().sum())
resumen["catalogo_regional_no_secuencia_validada"]=True

## Reglas decididas para plata

- **Incorporar como contexto sísmico y base de eventos**, no como capa de intensidad por comuna.
- Extraer `evento_id` desde el nivel Feature, magnitud con su tipo, profundidad desde Z en km y fechas UTC desde milisegundos; conservar JSON original y metadatos de consulta.
- Aunque la consulta pide M≥2,5, revisar el mínimo realmente observado y no afirmar completitud de eventos pequeños.
- Eventos posteriores en un radio no son automáticamente réplicas del principal: la asociación requiere evidencia de secuencia o criterio sismológico explícito.
- `mmi`/`cdi` del catálogo no son intensidad local por edificio. Para contrastar Cali con otras ciudades, adquirir ShakeMap/DYFI y evaluar cobertura antes del cruce espacial.
- El extracto regional actual no es un banco mundial de terremotos comparables.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('17_usgs_eventos_actualizados', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")